<a href="https://colab.research.google.com/github/CarlosJB95/PathIA-MSI-colon/blob/main/Notebook_02_numpy_imagen.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# **Semana 2 · Día 6 — NumPy sobre parches H&E**
Objetivo: tratar un parche como ndarray (alto, ancho, 3): crear, indexar, cortar, operar.
Cierra el Notebook 1 (Python intermedio). Arrastre a reabsorber hoy en la tarde: try/except + escribir manifest a CSV.

In [ ]:
import numpy as np
np.random.seed(42)   # misma "aleatoriedad" siempre → reproducible

In [ ]:
a = np.array([200], dtype=np.uint8)
print(a + 100)     # ¿300? No... → 44   (200+100 = 300; 300 - 256 = 44)

b = np.array([10], dtype=np.uint8)
print(b - 50)      # ¿-40? No... → 216  (da la vuelta por abajo)

In [ ]:
np.array([1, 2, 3])        # a partir de datos que YA tienes (una lista)
np.zeros((8, 8, 3))        # lienzo en blanco: todo 0.0  → reservar espacio
np.ones((8, 8, 3))         # todo 1.0                     → máscaras, pesos
np.arange(0, 10, 2)        # rango con paso → [0 2 4 6 8]  (como range de Python)
np.linspace(0, 1, 5)       # N puntos equiespaciados → [0. 0.25 0.5 0.75 1.]

In [ ]:
x = np.zeros((8, 8, 3))
print("shape:", x.shape)   # (8, 8, 3)  → tamaño por eje. TU dato más importante.
print("ndim :", x.ndim)    # 3          → número de ejes (dimensiones)
print("dtype:", x.dtype)   # float64    → tipo de cada elemento (default al no especificar)
print("size :", x.size)    # 192        → total de elementos = 8×8×3

In [ ]:
parche = np.random.randint(0, 256, size=(8, 8, 3), dtype=np.uint8)
print(parche.shape, parche.ndim, parche.dtype)   # (8, 8, 3) 3 uint8  ← tupla; uint8 porque lo fijé en el constructor

In [ ]:
print("R medio:", parche[:, :, 0].mean())
print("G medio:", parche[:, :, 1].mean())
print("B medio:", parche[:, :, 2].mean())

In [ ]:
# Parche 8x8: mitad izquierda "citoplasma" (eosina), mitad derecha "núcleo" (hematoxilina)
parche_he = np.zeros((8, 8, 3), dtype=np.uint8)

# Eosina (rosa): R alto, G/B medios-bajos  -> columnas 0-3
parche_he[:, 0:4] = [230, 130, 180]

# Hematoxilina (azul-morado): B alto, R medio, G bajo -> columnas 4-7
parche_he[:, 4:8] = [110, 90, 200]

# Media por canal de CADA región
print("Eosina  (izq):", parche_he[:, 0:4].mean(axis=(0,1)))
print("Hematox (der):", parche_he[:, 4:8].mean(axis=(0,1)))

In [ ]:
print(parche_he.mean().shape or "escalar")   # ()  → escalar, sin ejes
print(parche_he.mean(axis=(0,1)).shape)       # (3,)
print(parche_he.mean(axis=2).shape)           # (8, 8)

In [ ]:
p = np.zeros((256, 256, 3), dtype=np.uint8)
print("entrada:   ", p.shape)              # (256, 256, 3)
print("axis=(0,1):", p.mean(axis=(0,1)).shape)   # tacho 0 y 1 → (3, )
print("axis=2:    ", p.mean(axis=2).shape)       # tacho 2     → (256, 256)
print("axis=0:    ", p.mean(axis=0).shape)       # tacho 0     → (256, 3)

In [ ]:
parche_f = parche_he.astype(np.float32)      # a float primero (¿por qué? → q3 😉)
firma    = parche_f.mean(axis=(0,1))         # (3,)  → [media_R, media_G, media_B]

centrado = parche_f - firma                  # (8,8,3) - (3,)  ... ¿y funciona?
print(centrado.shape)                        # (8, 8, 3)

In [ ]:
a = np.ones((8, 8, 3))
print((a * 2).shape)               # (i)   (8, 8, 3) — un escalar cambia los VALORES, no la forma
print((a - np.ones((3,))).shape)   # (ii)  (8, 8, 3) — resta por canal (último eje 3 == 3)
# a - np.ones((8,))                # (iii) truena: último eje 3 vs 8, y ninguno es 1

In [ ]:
# 2 de las 8 filas = fondo (vidrio, casi blanco)
parche_he[0:2, :] = [245, 245, 245]

gris = parche_he.mean(axis=2)            # (a) (8, 8) → colapsa canales, conserva la rejilla espacial
mask_tejido = gris < 220                 # (b) bool  → máscara: True = tejido
pct_tejido = mask_tejido.mean() * 100    # (c) media de un booleano (True=1) = fracción de píxeles de tejido

print("shape máscara:", mask_tejido.shape)
print("% tejido:", pct_tejido)

In [ ]:
parche_f = parche_he.astype(np.float32)          # calcular en float (¿por qué? 😉)

print("media  RGB:", parche_f.mean(axis=(0,1)))  # firma de color
print("std    RGB:", parche_f.std(axis=(0,1)))   # dispersión por canal
print("min    RGB:", parche_f.min(axis=(0,1)))
print("max    RGB:", parche_f.max(axis=(0,1)))

# percentiles sobre UN canal (el rojo), para mediana e IQR
R = parche_f[:, :, 0]
p25, p50, p75 = np.percentile(R, [25, 50, 75])
print(f"\nCanal R → mediana={p50:.0f}, IQR={p75-p25:.0f}")

In [ ]:
def describir_parche(patch, slide_id, patch_id):
    """Extrae las métricas de un parche H&E y las devuelve como dict (una fila del manifest)."""
    gris = patch.mean(axis=2)
    pct_tejido = float((gris < 220).mean() * 100)      # ← float() añadido: cruza la aduana
    media_rgb = patch.astype(np.float32).mean(axis=(0, 1))
    return {
        "slide_id": slide_id,
        "patch_id": patch_id,
        "pct_tejido": round(pct_tejido, 1),
        "media_R": round(float(media_rgb[0]), 1),
        "media_G": round(float(media_rgb[1]), 1),
        "media_B": round(float(media_rgb[2]), 1),
    }

fila = describir_parche(parche_he, slide_id="TCGA-XX-0001", patch_id="p000")
print(fila)

In [ ]:
import csv

def escribir_manifest(filas, ruta):
    """Escribe una lista de dicts (filas del manifest) a un CSV. Maneja errores de E/S sin abortar."""
    if not filas:
        print("⚠️  No hay filas que escribir.")
        return False
    columnas = filas[0].keys()
    try:
        with open(ruta, "w", newline="") as f:
            writer = csv.DictWriter(f, fieldnames=columnas)
            writer.writeheader()
            writer.writerows(filas)
    except (OSError, PermissionError) as e:
        print(f"❌ No se pudo escribir '{ruta}': {e}")
        return False
    else:
        print(f"✅ Manifest escrito: {ruta}  ({len(filas)} filas)")
        return True

In [ ]:
# 3 filas: el parche real + dos variantes con distinto % de fondo
p_mixto = parche_he.copy()
p_fondo = parche_he.copy();  p_fondo[:6, :] = [245, 245, 245]   # casi todo fondo

filas = [
    describir_parche(p_mixto, "TCGA-XX-0001", "p000"),
    describir_parche(p_fondo, "TCGA-XX-0001", "p001"),
]

escribir_manifest(filas, "manifest_parches.csv")

In [ ]:
# 1) Caso bueno
escribir_manifest(filas, "manifest_parches.csv")        # ✅ ... (2 filas)

# 2) Provoca el fallo a propósito
escribir_manifest(filas, "/carpeta_inexistente/manifest.csv")   # ❌ ... y NO muere

print("El programa sigue vivo después del error 👇")     # ← esta línea DEBE ejecutarse

## **Semana 2 · Día 7 — Operaciones vectorizadas y estadística**

Temas: Reducciones por eje, lógica booleana, np.where y estadística descriptiva (media/mediana/SD/percentiles/IQR/histograma) sobre el parche H&E

In [ ]:
R = parche_he[:, :, 0].astype(np.int16)   # int16 para restar sin desborde de uint8
B = parche_he[:, :, 2].astype(np.int16)

es_tejido  = gris < 220        # ya la conoces: oscuro
es_azulado = B > R             # el canal azul supera al rojo → hematoxilina

nucleos = es_tejido & es_azulado    # tejido Y azulado

In [ ]:
nucleos = (gris < 220) & (B > R)
print(nucleos.shape, nucleos.dtype)
print("% tejido :", (gris < 220).mean() * 100)
print("% núcleos:", nucleos.mean() * 100)

In [ ]:
# Resaltar núcleos: 255 donde hay núcleo, 0 donde no → imagen binaria
resaltado = np.where(nucleos, 255, 0)
print(resaltado.shape, resaltado.dtype)
print(resaltado)

In [ ]:
# Versión 2D (binaria) — pregunta 1
resaltado = np.where(nucleos, 255, 0)
print("binaria:", resaltado.shape)          # (8, 8)

# Versión a color — pregunta 2, ya arreglada
verde = np.array([0, 255, 0], dtype=np.uint8)
coloreado = np.where(nucleos[:, :, None], verde, parche_he)
print("color:  ", coloreado.shape)          # (8, 8, 3)

In [ ]:
def estadisticas_parche(patch, umbral_tejido=220):
    """Describe un parche H&E: % tejido, % núcleos y firma de color por canal.
    Devuelve un dict (una fila de manifest). Solo calcula; no toca disco."""
    gris = patch.mean(axis=2)
    R = patch[:, :, 0].astype(np.int16)
    B = patch[:, :, 2].astype(np.int16)

    es_tejido = gris < umbral_tejido
    nucleos   = es_tejido & (B > R)
    media_rgb = patch.astype(np.float32).mean(axis=(0, 1))

    return {
        "pct_tejido":  round(float(es_tejido.mean() * 100), 1),
        "pct_nucleos": round(float(nucleos.mean() * 100), 1),
        "media_R":     round(float(media_rgb[0]), 1),
        "media_G":     round(float(media_rgb[1]), 1),
        "media_B":     round(float(media_rgb[2]), 1),
    }

print(estadisticas_parche(parche_he))

In [ ]:
def pct_tejido_lento(patch, umbral=220):
    """Versión con bucles: recorre cada píxel a mano. LENTA a propósito."""
    H, W, _ = patch.shape
    cuenta = 0
    for i in range(H):
        for j in range(W):
            gris_px = (int(patch[i,j,0]) + int(patch[i,j,1]) + int(patch[i,j,2])) / 3
            if gris_px < umbral:
                cuenta += 1
    return cuenta / (H * W) * 100

# La versión vectorizada (una línea, todo NumPy):
def pct_tejido_rapido(patch, umbral=220):
    return (patch.mean(axis=2) < umbral).mean() * 100

In [ ]:
grande = np.random.randint(0, 256, size=(256, 256, 3), dtype=np.uint8)

# PASO 1 — ¿dan lo mismo? (corrección antes que velocidad)
print(pct_tejido_lento(grande), pct_tejido_rapido(grande))   # deben coincidir

In [ ]:
%%timeit
pct_tejido_lento(grande)

In [ ]:
%%timeit
pct_tejido_rapido(grande)

In [ ]:
gris = parche.mean(axis=2)
pct_fondo = (gris > 230).mean() * 100
print(round(float(pct_fondo), 1))

In [ ]:
def filtrar_parche(patch, umbral=220):
    """Describe un parche H&E y decide si se conserva. Solo calcula; no toca disco."""
    gris = patch.mean(axis=2)
    es_tejido = gris < umbral
    pct = float(es_tejido.mean() * 100)
    m = patch.astype(np.float32).mean(axis=(0, 1))
    return {
        "pct_tejido": round(pct, 1),
        "media_R": round(float(m[0]), 1),
        "media_G": round(float(m[1]), 1),
        "media_B": round(float(m[2]), 1),
        "conservar": pct >= 50,
    }

print(filtrar_parche(parche))

In [ ]:
# 3 parches con distinto % de fondo, para variar
p0 = parche.copy()
p1 = parche.copy(); p1[:5, :] = 245     # mayoría fondo
p2 = parche.copy(); p2[:2, :] = 245     # poco fondo

filas = [filtrar_parche(p, umbral=220) for p in [p0, p1, p2]]
for f in filas:
    print(f)